# SolarScan · Hito 2 · Serie temporal de generación

**Objetivo.** Detectar inversores que generan menos energía de la esperada para la irradiancia
del día. Una caída sostenida del rendimiento indica suciedad, sombra o una falla, y es la señal
que complementa a la inspección fotográfica: la serie dice *cuánto* se está perdiendo y la foto
dice *por qué*.

**Datos.** *Solar Power Generation Data* (Kaggle, autor Ani Kannal), planta 2: 22 inversores con
registros cada 15 minutos entre el 15 de mayo y el 17 de junio de 2020, más un sensor de
irradiancia y temperatura. Los archivos originales se guardan en `serie_temporal/datos/`.

**Modelos comparados** (mismo período de prueba, mismas métricas):

| Tipo | Modelo | Idea |
|---|---|---|
| Simple | Ingenuo (persistencia) | La potencia esperada es la del mismo intervalo del día anterior |
| Simple | Promedio móvil de 3 días | Promedio del mismo intervalo en los tres días anteriores |
| Clásico | Regresión lineal física | Potencia = a·G + b·G·(Tmódulo − 25) + c |
| Deep Learning | LSTM | Ventana de 2 horas de irradiancia, temperaturas y hora del día |

**División temporal:** entrenamiento 15-may a 3-jun, validación 4-jun a 10-jun, prueba 11-jun a 17-jun.

In [1]:
import json, random, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
from torch import nn

SEMILLA = 42
random.seed(SEMILLA); np.random.seed(SEMILLA); torch.manual_seed(SEMILLA)
torch.use_deterministic_algorithms(True)

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATOS = RAIZ / "serie_temporal" / "datos"
ART = RAIZ / "serie_temporal" / "artefactos"
ART.mkdir(parents=True, exist_ok=True)
FIG = RAIZ / "notebooks" / "figuras"
FIG.mkdir(parents=True, exist_ok=True)

ENTRENAMIENTO = ("2020-05-15", "2020-06-03")
VALIDACION = ("2020-06-04", "2020-06-10")
PRUEBA = ("2020-06-11", "2020-06-17")
G_MIN = 0.02          # irradiancia mínima (kW/m2) para considerar un intervalo diurno
VENTANA = 8           # 8 intervalos de 15 min = 2 horas de contexto para la LSTM

## 1. Carga y limpieza

In [2]:
gen = pd.read_csv(DATOS / "Plant_2_Generation_Data.csv.gz", parse_dates=["DATE_TIME"])
clima = pd.read_csv(DATOS / "Plant_2_Weather_Sensor_Data.csv.gz", parse_dates=["DATE_TIME"])
print("Generación:", gen.shape, "| Clima:", clima.shape)
print("Duplicados generación:", gen.duplicated(["DATE_TIME", "SOURCE_KEY"]).sum(),
      "| Duplicados clima:", clima.duplicated("DATE_TIME").sum())

slots = pd.date_range(gen.DATE_TIME.min(), gen.DATE_TIME.max(), freq="15min")
inversores = sorted(gen.SOURCE_KEY.unique())
ALIAS = {k: f"INV-{i+1:02d}" for i, k in enumerate(inversores)}
esperadas = len(slots) * len(inversores)
print(f"Registros esperados: {esperadas} | presentes: {len(gen)} | faltantes: {esperadas-len(gen)} "
      f"({100*(esperadas-len(gen))/esperadas:.1f} %)")

# Malla completa inversor x intervalo; los huecos quedan como NaN y no se imputan.
malla = pd.MultiIndex.from_product([slots, inversores], names=["DATE_TIME", "SOURCE_KEY"]).to_frame(index=False)
df = malla.merge(gen[["DATE_TIME", "SOURCE_KEY", "AC_POWER"]], how="left")
df = df.merge(clima[["DATE_TIME", "IRRADIATION", "MODULE_TEMPERATURE", "AMBIENT_TEMPERATURE"]], how="left")
df["fecha"] = df.DATE_TIME.dt.normalize()
df["inversor"] = df.SOURCE_KEY.map(ALIAS)
print("Intervalos sin clima:", df.IRRADIATION.isna().groupby(df.DATE_TIME).first().sum())

Generación: (67698, 7) | Clima: (3259, 6)
Duplicados generación: 0 | Duplicados clima: 0
Registros esperados: 71808 | presentes: 67698 | faltantes: 4110 (5.7 %)
Intervalos sin clima: 5


### Días sanos para entrenar
Los modelos aprenden cómo debería generar un inversor **sano**. Para no enseñarles las fallas,
se marca como sano un día-inversor si su rendimiento (energía / irradiancia) está entre 0,9 y
1,1 veces la mediana de los 22 inversores ese mismo día y si tiene al menos 90 % de los
intervalos diurnos registrados. El filtro se usa para entrenar y para la evaluación de error;
la detección de anomalías se evalúa en todos los días.

In [3]:
dia = df[df.IRRADIATION > G_MIN]
agg = dia.groupby(["inversor", "fecha"]).agg(e=("AC_POWER", "sum"), g=("IRRADIATION", "sum"),
                                              n=("AC_POWER", "count"), n_tot=("IRRADIATION", "size"))
agg["rend"] = agg.e / agg.g
agg["ratio_pares"] = agg.rend / agg.groupby("fecha").rend.transform("median")
agg["completo"] = agg.n / agg.n_tot
agg["sano"] = agg.ratio_pares.between(0.9, 1.1) & (agg.completo >= 0.9)
print("Días-inversor sanos:", int(agg.sano.sum()), "de", len(agg))
df = df.merge(agg[["sano", "ratio_pares"]].reset_index(), on=["inversor", "fecha"], how="left")
df["sano"] = df.sano.fillna(False).astype(bool)

def en(rango, d=None):
    d = df if d is None else d
    return d[(d.fecha >= rango[0]) & (d.fecha <= rango[1])]

# Capacidad de referencia de cada inversor (percentil 99 de su potencia en días sanos de entrenamiento).
cap = en(ENTRENAMIENTO)[lambda d: d.sano].groupby("inversor").AC_POWER.quantile(0.99)
df["cap"] = df.inversor.map(cap)
df["y"] = df.AC_POWER / df.cap
hora = df.DATE_TIME.dt.hour + df.DATE_TIME.dt.minute / 60
df["h_sin"] = np.sin(2 * np.pi * hora / 24)
df["h_cos"] = np.cos(2 * np.pi * hora / 24)
print(cap.describe().round(1))

Días-inversor sanos: 341 de 748
count      22.0
mean     1225.5
std        67.0
min       977.9
25%      1209.1
50%      1247.4
75%      1263.1
max      1295.4
Name: AC_POWER, dtype: float64


## 2. Modelos

In [4]:
# 2.1 Modelos simples: persistencia y promedio móvil (mismo intervalo, días anteriores).
df = df.sort_values(["inversor", "DATE_TIME"]).reset_index(drop=True)
df["slot"] = df.DATE_TIME.dt.hour * 4 + df.DATE_TIME.dt.minute // 15
piv = df.pivot_table(index=["inversor", "slot"], columns="fecha", values="y")
fechas = list(piv.columns)
ingenuo = piv.shift(1, axis=1)
movil = piv.T.rolling(3, min_periods=1).mean().shift(1).T
df["pred_ingenuo"] = ingenuo.stack(future_stack=True).reindex(pd.MultiIndex.from_frame(df[["inversor", "slot", "fecha"]])).to_numpy()
df["pred_movil"] = movil.stack(future_stack=True).reindex(pd.MultiIndex.from_frame(df[["inversor", "slot", "fecha"]])).to_numpy()

# 2.2 Modelo clásico: regresión lineal física sobre días sanos de entrenamiento.
def X_reg(d):
    G = d.IRRADIATION.to_numpy(); T = d.MODULE_TEMPERATURE.to_numpy()
    return np.column_stack([G, G * (T - 25), np.ones_like(G)])

tr = en(ENTRENAMIENTO)[lambda d: d.sano & (d.IRRADIATION > G_MIN) & d.y.notna()]
coef, *_ = np.linalg.lstsq(X_reg(tr), tr.y.to_numpy(), rcond=None)
print("Coeficientes regresión [a, b, c]:", np.round(coef, 4))
ok = df.IRRADIATION.notna()
df["pred_regresion"] = np.nan
df.loc[ok, "pred_regresion"] = np.clip(X_reg(df[ok]) @ coef, 0, None)
df.loc[df.IRRADIATION <= G_MIN, "pred_regresion"] = 0.0

Coeficientes regresión [a, b, c]: [ 1.0959 -0.0094  0.0369]


In [5]:
# 2.3 LSTM: ventana de 2 horas de variables meteorológicas -> potencia normalizada en t.
CARACT = ["IRRADIATION", "MODULE_TEMPERATURE", "AMBIENT_TEMPERATURE", "h_sin", "h_cos"]
clima_t = df.drop_duplicates("DATE_TIME").set_index("DATE_TIME")[CARACT].sort_index()
mu = clima_t[CARACT[:3]].mean(); sd = clima_t[CARACT[:3]].std()
clima_n = clima_t.copy()
clima_n[CARACT[:3]] = (clima_t[CARACT[:3]] - mu) / sd
clima_n = clima_n.ffill().fillna(0)   # 5 intervalos sin clima; solo afecta el contexto de la ventana
idx_t = {t: i for i, t in enumerate(clima_n.index)}
M = clima_n.to_numpy(np.float32)

def ventanas(d):
    pos = np.array([idx_t[t] for t in d.DATE_TIME])
    pos = np.clip(pos, VENTANA - 1, None)
    return np.stack([M[p - VENTANA + 1:p + 1] for p in pos])

class LSTMSolar(nn.Module):
    def __init__(self, n_in=len(CARACT), oculto=32):
        super().__init__()
        self.lstm = nn.LSTM(n_in, oculto, batch_first=True)
        self.salida = nn.Sequential(nn.Linear(oculto, 16), nn.ReLU(), nn.Linear(16, 1))
    def forward(self, x):
        h, _ = self.lstm(x)
        return self.salida(h[:, -1]).squeeze(-1)

def datos_lstm(rango):
    d = en(rango)[lambda d: d.sano & (d.IRRADIATION > G_MIN) & d.y.notna()]
    return torch.tensor(ventanas(d)), torch.tensor(d.y.to_numpy(np.float32))

Xtr, ytr = datos_lstm(ENTRENAMIENTO); Xva, yva = datos_lstm(VALIDACION)
print("Muestras LSTM entrenamiento/validación:", len(ytr), len(yva))

def entrenar_lstm(oculto, lr, epocas=80, paciencia_max=10):
    """Entrena una LSTM y devuelve (mse_val, mejor_epoca, historia, estado)."""
    torch.manual_seed(SEMILLA)
    red = LSTMSolar(oculto=oculto)
    opt = torch.optim.Adam(red.parameters(), lr=lr)
    perdida = nn.MSELoss()
    g = torch.Generator().manual_seed(SEMILLA)
    mejor, mejor_epoca, historia, paciencia, estado = 1e9, 0, [], 0, None
    for epoca in range(1, epocas + 1):
        red.train()
        orden = torch.randperm(len(ytr), generator=g)
        for i in range(0, len(ytr), 256):
            b = orden[i:i + 256]
            opt.zero_grad(); l = perdida(red(Xtr[b]), ytr[b]); l.backward(); opt.step()
        red.eval()
        with torch.no_grad():
            lv = perdida(red(Xva), yva).item(); lt = perdida(red(Xtr), ytr).item()
        historia.append({"epoca": epoca, "train_mse": lt, "val_mse": lv})
        if lv < mejor - 1e-6:
            mejor, mejor_epoca, paciencia = lv, epoca, 0
            estado = {k: v.clone() for k, v in red.state_dict().items()}
        else:
            paciencia += 1
            if paciencia >= paciencia_max: break
    return mejor, mejor_epoca, historia, estado

# Búsqueda pequeña de hiperparámetros: se elige por MSE de validación (la prueba no se mira).
t0 = time.perf_counter()
busqueda = []
for oculto in (16, 32, 64):
    for lr in (3e-3, 1e-3):
        mse, ep, hist, est = entrenar_lstm(oculto, lr)
        busqueda.append({"oculto": oculto, "lr": lr, "mse_val": mse, "epoca": ep, "hist": hist, "estado": est})
tabla_busqueda = pd.DataFrame([{k: v for k, v in b.items() if k not in ("hist", "estado")} for b in busqueda])
print(tabla_busqueda.round(5).to_string(index=False))
elegida = min(busqueda, key=lambda b: b["mse_val"])
OCULTO, mejor, mejor_epoca, historia = elegida["oculto"], elegida["mse_val"], elegida["epoca"], elegida["hist"]
red = LSTMSolar(oculto=OCULTO); red.load_state_dict(elegida["estado"])
torch.save(red.state_dict(), ART / "lstm_solar.pt")
print(f"Búsqueda LSTM: {time.perf_counter()-t0:.1f} s · elegida oculto={OCULTO}, lr={elegida['lr']}, "
      f"época {mejor_epoca}, MSE val {mejor:.5f}")
# Referencia: MSE de validación de la regresión, en la misma escala.
va_reg = en(VALIDACION)[lambda d: d.sano & (d.IRRADIATION > G_MIN) & d.y.notna()]
print("MSE val regresión:", round(float(np.mean((np.clip(X_reg(va_reg) @ coef, 0, None) - va_reg.y) ** 2)), 5))
red.load_state_dict(torch.load(ART / "lstm_solar.pt", weights_only=True)); red.eval()

mask = (df.IRRADIATION > G_MIN) & df.IRRADIATION.notna()
with torch.no_grad():
    df["pred_lstm"] = 0.0
    df.loc[mask, "pred_lstm"] = np.clip(red(torch.tensor(ventanas(df[mask]))).numpy(), 0, None)
df.loc[df.IRRADIATION.isna(), "pred_lstm"] = np.nan

h = pd.DataFrame(historia)
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(h.epoca, h.train_mse, label="entrenamiento"); ax.plot(h.epoca, h.val_mse, label="validación")
ax.set_xlabel("época"); ax.set_ylabel("MSE (potencia normalizada)"); ax.set_title("Curva de aprendizaje LSTM"); ax.legend()
fig.tight_layout(); fig.savefig(FIG / "serie_curva_lstm.png", dpi=150)

Muestras LSTM entrenamiento/validación: 6813 3812


 oculto    lr  mse_val  epoca
     16 0.003  0.02396     11
     16 0.001  0.02404     11
     32 0.003  0.02348      7
     32 0.001  0.02659      7
     64 0.003  0.02450      7
     64 0.001  0.02683      6
Búsqueda LSTM: 12.1 s · elegida oculto=32, lr=0.003, época 7, MSE val 0.02348
MSE val regresión: 0.02069


## 3. Error de estimación en días sanos de prueba

In [6]:
MODELOS = {"Ingenuo (día anterior)": "pred_ingenuo", "Promedio móvil 3 días": "pred_movil",
           "Regresión lineal (clásico)": "pred_regresion", "LSTM (Deep Learning)": "pred_lstm"}
pr = en(PRUEBA)[lambda d: d.sano & (d.IRRADIATION > G_MIN) & d.y.notna()]
filas = []
for nombre, col in MODELOS.items():
    p = pr.dropna(subset=[col])
    err = (p[col] - p.y) * p.cap               # error en kW
    e_dia = p.assign(r=p[col] * p.cap).groupby(["inversor", "fecha"]).agg(r=("r", "sum"), a=("AC_POWER", "sum"))
    filas.append({"modelo": nombre, "MAE_kW": err.abs().mean(), "RMSE_kW": np.sqrt((err ** 2).mean()),
                  "error_energia_diaria_%": (100 * (e_dia.r - e_dia.a).abs() / e_dia.a).mean(),
                  "n": len(p)})
tabla_error = pd.DataFrame(filas).set_index("modelo").round(2)
print(tabla_error)

                            MAE_kW  RMSE_kW  error_energia_diaria_%     n
modelo                                                                   
Ingenuo (día anterior)      208.89   303.54                   25.69  5615
Promedio móvil 3 días       211.35   287.63                   31.90  5615
Regresión lineal (clásico)   71.09   128.68                    4.01  5615
LSTM (Deep Learning)         86.24   145.80                    5.08  5615


## 4. Detección de anomalías

Para cada inversor y día se calcula el **índice de rendimiento** = energía real / energía esperada.
Si baja de un umbral, el día se marca como anomalía. Como el dataset no trae etiquetas de falla,
la detección se evalúa con **fallas inyectadas** con supuestos explícitos: sobre días sanos se
multiplica la potencia por 0,80, 0,60 o 0,30 durante **dos días seguidos** (suciedad o falla que
persiste). Se considera falla toda pérdida de 15 % o más. El umbral se elige en validación y se
aplica sin cambios en prueba.

In [7]:
def indice_dia(d, col, factor=None):
    """Energía real / esperada por inversor y día (solo intervalos diurnos)."""
    d = d[(d.IRRADIATION > G_MIN)].dropna(subset=[col, "AC_POWER"])
    real = d.AC_POWER if factor is None else d.AC_POWER * d.fecha.map(factor).fillna(1.0)
    return (real.groupby([d.inversor, d.fecha]).sum() /
            (d[col] * d.cap).groupby([d.inversor, d.fecha]).sum())

def escenarios(rango, rng):
    """Por inversor: un escenario por factor con 2 días seguidos sanos que se degradan."""
    casos = []
    dias = [pd.Timestamp(x) for x in sorted(en(rango).fecha.unique())]
    sanos = set(agg.index[agg.sano])
    for inv in sorted(df.inversor.unique()):
        cand = [i for i in range(1, len(dias)) if (inv, dias[i - 1]) in sanos and (inv, dias[i]) in sanos]
        for f in (0.80, 0.60, 0.30):
            if not cand: continue
            i = int(rng.choice(cand))
            casos.append((inv, {dias[i - 1]: f, dias[i]: f}))
    return casos

def evaluar_deteccion(rango, umbral, rng_seed=SEMILLA):
    rng = np.random.default_rng(rng_seed)
    casos = escenarios(rango, rng)
    res = {}
    for nombre, col in MODELOS.items():
        y_true, y_pred = [], []
        for inv, falla in casos:
            d = en(rango)[lambda x: x.inversor == inv].copy()
            if col in ("pred_ingenuo", "pred_movil"):
                # Los modelos simples usan la potencia observada de días previos: si el día anterior
                # ya estaba degradado, su referencia también lo está.
                d = df[df.inversor == inv].copy()
                d["AC_POWER"] = d.AC_POWER * d.fecha.map(falla).fillna(1.0)
                yy = d.AC_POWER / d.cap
                p = d.assign(y=yy).pivot_table(index="slot", columns="fecha", values="y")
                ref = p.shift(1, axis=1) if col == "pred_ingenuo" else p.T.rolling(3, min_periods=1).mean().shift(1).T
                d[col] = ref.stack(future_stack=True).reindex(pd.MultiIndex.from_frame(d[["slot", "fecha"]])).to_numpy()
                d = en(rango, d)
                ir = indice_dia(d, col)
            else:
                ir = indice_dia(d, col, falla)
            for (iv, fe), v in ir.items():
                fe = pd.Timestamp(fe)
                if not (fe in falla or bool(agg.sano.get((inv, fe), False))): continue   # se evalúan días sanos y degradados
                y_true.append(fe in falla); y_pred.append(bool(v < umbral))
        y_true, y_pred = np.array(y_true, dtype=bool), np.array(y_pred, dtype=bool)
        tp = (y_true & y_pred).sum(); fp = (~y_true & y_pred).sum(); fn = (y_true & ~y_pred).sum()
        prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
        res[nombre] = {"precision": prec, "recall": rec, "f1": 2 * prec * rec / max(prec + rec, 1e-9),
                       "falsas_alarmas": int(fp), "fallas_no_detectadas": int(fn), "n_dias": len(y_true)}
    return pd.DataFrame(res).T

# Elección del umbral en validación con el modelo que se usará en producción y con los demás.
umbrales = [0.75, 0.80, 0.85, 0.90, 0.95]
sel = {u: evaluar_deteccion(VALIDACION, u) for u in umbrales}
f1_val = pd.DataFrame({u: sel[u].f1 for u in umbrales}).round(3)
print("F1 en validación por umbral:\n", f1_val)

F1 en validación por umbral:
                              0.75   0.80   0.85   0.90   0.95
Ingenuo (día anterior)      0.508  0.537  0.537  0.613  0.629
Promedio móvil 3 días       0.646  0.691  0.746  0.776  0.774
Regresión lineal (clásico)  0.809  0.841  0.880  0.950  0.912
LSTM (Deep Learning)        0.817  0.833  0.849  0.937  0.911


### Selección del modelo y del umbral
Se elige el modelo con mejor F1 de detección en validación y, para ese modelo, el umbral con
mejor F1 (en empate, el más alto, porque deja pasar menos fallas).

In [8]:
mejor_modelo = f1_val.max(axis=1).idxmax()
fila = f1_val.loc[mejor_modelo]
UMBRAL = float(max(u for u in umbrales if fila[u] == fila.max()))
print("Modelo elegido:", mejor_modelo, "| umbral:", UMBRAL)
tabla_det = evaluar_deteccion(PRUEBA, UMBRAL).round(3)
print("Detección en PRUEBA (umbral elegido en validación):\n", tabla_det)

Modelo elegido: Regresión lineal (clásico) | umbral: 0.9


Detección en PRUEBA (umbral elegido en validación):
                             precision  recall     f1  falsas_alarmas  \
Ingenuo (día anterior)          0.491   0.658  0.562            82.0   
Promedio móvil 3 días           0.572   0.792  0.664            71.0   
Regresión lineal (clásico)      0.984   1.000  0.992             2.0   
LSTM (Deep Learning)            1.000   0.975  0.987             0.0   

                            fallas_no_detectadas  n_dias  
Ingenuo (día anterior)                      41.0   339.0  
Promedio móvil 3 días                       25.0   339.0  
Regresión lineal (clásico)                   0.0   339.0  
LSTM (Deep Learning)                         3.0   339.0  


## 5. Anomalías reales detectadas en la prueba

In [9]:
col_prod = MODELOS[mejor_modelo]
ir_real = indice_dia(en(PRUEBA), col_prod).rename("indice").reset_index()
ir_real["anomalia"] = ir_real.indice < UMBRAL
d_test = en(PRUEBA)[lambda d: d.IRRADIATION > G_MIN].dropna(subset=[col_prod, "AC_POWER"])
energia = d_test.assign(esp=d_test[col_prod] * d_test.cap).groupby(["inversor", "fecha"]).agg(
    real_kwh=("AC_POWER", lambda s: s.sum() * 0.25), esperada_kwh=("esp", lambda s: s.sum() * 0.25)).reset_index()
ir_real = ir_real.merge(energia, on=["inversor", "fecha"])
ir_real["perdida_kwh"] = (ir_real.esperada_kwh - ir_real.real_kwh).clip(lower=0) * ir_real.anomalia
print("Días-inversor en prueba:", len(ir_real), "| marcados como anomalía:", int(ir_real.anomalia.sum()))
print("Energía no generada en días marcados (kWh):", round(ir_real.perdida_kwh.sum(), 0))
print(ir_real.sort_values("indice").head(10).round(2).to_string(index=False))

# Figura: real vs esperado en un inversor con anomalía real.
ej = ir_real.sort_values("indice").iloc[0]
d_ej = df[(df.inversor == ej.inversor) & (df.fecha >= pd.Timestamp(ej.fecha) - pd.Timedelta(days=2)) &
          (df.fecha <= pd.Timestamp(ej.fecha) + pd.Timedelta(days=1))]
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(d_ej.DATE_TIME, d_ej.AC_POWER, label="real", color="#1F2937")
ax.plot(d_ej.DATE_TIME, d_ej.pred_lstm * d_ej.cap, label="esperado LSTM", color="#0F9D8A")
ax.plot(d_ej.DATE_TIME, d_ej.pred_regresion * d_ej.cap, label="esperado regresión", color="#E39B2D", ls="--")
ax.set_ylabel("Potencia AC (kW)"); ax.set_title(f"{ej.inversor}: potencia real vs esperada")
ax.legend(); fig.autofmt_xdate(); fig.tight_layout(); fig.savefig(FIG / "serie_ejemplo_anomalia.png", dpi=150)

Días-inversor en prueba: 154 | marcados como anomalía: 31
Energía no generada en días marcados (kWh): 94740.0
inversor      fecha  indice  anomalia  real_kwh  esperada_kwh  perdida_kwh
  INV-11 2020-06-14    0.30      True   2108.91       6945.72      4836.82
  INV-01 2020-06-14    0.31      True   2165.77       6989.30      4823.53
  INV-04 2020-06-14    0.33      True   2048.13       6234.06      4185.93
  INV-11 2020-06-15    0.37      True   2230.92       5977.45      3746.53
  INV-04 2020-06-15    0.41      True   2203.07       5365.00      3161.92
  INV-01 2020-06-15    0.41      True   2495.65       6014.95      3519.30
  INV-12 2020-06-14    0.42      True   2276.90       5380.67      3103.77
  INV-03 2020-06-13    0.42      True   2763.10       6510.38      3747.29
  INV-13 2020-06-15    0.44      True   2563.14       5889.71      3326.57
  INV-21 2020-06-13    0.44      True   3004.30       6800.76      3796.45


/tmp/ipykernel_394/2682877575.py:11: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(ir_real.sort_values("indice").head(10).round(2).to_string(index=False))


## 6. Exportación para el servicio
Se guardan los coeficientes de la regresión, la normalización, la capacidad por inversor, el
umbral y las métricas. El servicio (`serie_temporal/servicio_serie.py`) carga estos archivos y
ejecuta la LSTM y la regresión cada vez que el flujo lo consulta.

In [10]:
config = {
    "dataset": "Kaggle anikannal/solar-power-generation-data (Plant 2)",
    "periodo": [str(df.fecha.min().date()), str(df.fecha.max().date())],
    "division": {"entrenamiento": ENTRENAMIENTO, "validacion": VALIDACION, "prueba": PRUEBA},
    "g_min": G_MIN, "ventana": VENTANA, "caracteristicas": CARACT,
    "normalizacion": {"media": mu.to_dict(), "desv": sd.to_dict()},
    "capacidad_kw": cap.round(3).to_dict(),
    "alias": ALIAS,
    "regresion": {"coef": coef.tolist(), "forma": "y = a*G + b*G*(Tmod-25) + c"},
    "lstm": {"oculto": OCULTO, "lr": elegida["lr"], "mejor_epoca": mejor_epoca, "mse_validacion": mejor,
             "busqueda": tabla_busqueda.to_dict(orient="records")},
    "modelo_produccion": mejor_modelo, "umbral_indice": UMBRAL,
    "metricas_error_prueba": tabla_error.reset_index().to_dict(orient="records"),
    "metricas_deteccion_prueba": tabla_det.reset_index().rename(columns={"index": "modelo"}).to_dict(orient="records"),
    "f1_validacion_por_umbral": {m: {str(u): float(v) for u, v in r.items()} for m, r in f1_val.iterrows()},
    "anomalias_reales_prueba": {"dias_inversor": int(len(ir_real)), "marcados": int(ir_real.anomalia.sum()),
                                "perdida_kwh": float(round(ir_real.perdida_kwh.sum(), 1))},
}
(ART / "config_serie.json").write_text(json.dumps(config, indent=2, ensure_ascii=False, default=str))
print("Artefactos:", sorted(p.name for p in ART.iterdir()))

Artefactos: ['config_serie.json', 'lstm_solar.pt']
